Tester

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

n_rows = 100

df_clean = pd.DataFrame({
    "feature_a": np.random.normal(loc=0.0, scale=1.0, size=n_rows),
    "feature_b": np.random.uniform(low=0, high=10, size=n_rows),
    "feature_c": np.random.poisson(lam=5, size=n_rows),
    "feature_d": np.random.normal(loc=10.0, scale=2.0, size=n_rows),
    "feature_e": np.random.binomial(n=1, p=0.3, size=n_rows),
})

df_clean.head()

In [ ]:
n_error_rows = 20

# Sample rows from the clean dataframe
df_error = df_clean.sample(n_error_rows, random_state=1).reset_index(drop=True)

# --- Inject errors ---

# Additive constant error
df_error["feature_a"] = df_error["feature_a"] + 2.5

# Scaling error
df_error["feature_b"] = df_error["feature_b"] * 1.8

# Increased noise
df_error["feature_c"] = df_error["feature_c"] + np.random.normal(
    loc=0, scale=2.0, size=n_error_rows
)

# Missing values (randomly drop ~30%)
mask_missing = np.random.rand(n_error_rows) < 0.3
df_error.loc[mask_missing, "feature_d"] = np.nan

# Logical / data-entry error: flip binary with noise
flip_mask = np.random.rand(n_error_rows) < 0.2
df_error.loc[flip_mask, "feature_e"] = 1 - df_error.loc[flip_mask, "feature_e"]

df_error.head()


In [ ]:
# cleaner (uses inference)
import pandas as pd

from conformal_data_cleaning.cleaner.autogluon import ConformalAutoGluonCleaner

# Make cleaner
cleaner: ConformalAutoGluonCleaner = ConformalAutoGluonCleaner(confidence_level= 0.999, seed = 42)


# Do
fit_cleaner = cleaner.fit(df_clean)

cleaned_data: tuple[pd.DataFrame, pd.DataFrame] = fit_cleaner.transform(df_error)

In [ ]:
data, mask = cleaned_data

In [ ]:
(data != df_error).sum()

In [ ]:
mask.sum()